# Deney 5 — `ilac_genel8k_r32`: genel Türkçe'yi tavana çek

**Üç bulgu bu deneyi zorunlu kıldı:**

1. **Hata analizi** (`egitim/ilac_hata_analizi.py`): İlaç adı cümleden çıkarıldığında
   normalize WER **yükseliyor** (7.74% → 8.36%). Yani ilaç adları cümlenin geri kalanından
   daha iyi tanınıyor — kalan hataların ağırlığı **sıradan Türkçe'de**. İlaç adı gövde
   doğruluğu zaten %96.6.
2. **FLEURS tükendi**: `google/fleurs` `tr_tr` train+validation = **2864 satır, hepsi bu**.
   Deney 4 ~2500'ünü kullandı. Ölçeklenecek yer yok → bu deneyde **tamamını** kullanıyoruz.
3. **Common Voice'ta bol yer var**: 58.427 TÜİK-etiketli satırın sadece 7.580'i kullanıldı.
   ~50.800 satır boş → **8000'e çıkarıyoruz** (Deney 4'te 2500'dü).

**Tek değişken**: genel Türkçe verisinin miktarı. r=32/alpha=64, dropout, augmentation
ayarları Deney 4 ile **birebir aynı**.

**İlaç dozu sabit**: 3910 örnek × 12 epoch — ilaç örneklerinin gördüğü gradyan güncellemesi
sayısı değişmiyor, sadece genel Türkçe dozu artıyor. Bu yüzden ilaç performansının bozulma
riski düşük (Deney 2→3'te ilaç oranı %79.6'dan %43.9'a düşerken ilaç WER'i *iyileşmişti*).

| | Deney 4 | **Deney 5** |
|---|---|---|
| ilaç | 3910 | 3910 |
| **acil_tip** | **yok** | **505** |
| FLEURS | ~2500 | **2864 (tamamı)** |
| Common Voice | 2500 | **8000** |
| toplam | 8910 | **15.279** |
| genel Türkçe oranı | %56.1 | **%71.2** |
| adım/epoch | 557 | **955** |
| epoch | 15 | **12** |
| toplam adım | 8355 | **11.460** |
| LoRA | r=32 / alpha=64 | r=32 / alpha=64 (aynı) |

**acil_tip neden eklendi**: Projede 16 dal daha üretilecek, yani Deney 5 nihai model
değil **nihai metodoloji provası**. Provanın gelecekteki 18-dallı koşuya benzemesi için
**çok-alanlı + rehearsal** konfigürasyonunu test etmesi gerekir. Ayrıca eldeki tek
acil_tip modeli (`acil_tip_baseline`, r=64, rehearsal YOK) bilinen kötü tarifle
eğitilmiş ve genel Türkçe kaybı hiç ölçülmemiş — sevk edilebilir bir model değil.

acil_tip'in **val (96) + test (32) = 128 satırı eğitime GİRMEZ**, 4. değerlendirme
ekseni olarak ayrı tutulur (eski deneylerin 32'lik test setinden 4 kat az gürültülü).

**12 epoch neden**: Deney 4'ün val_loss'u ~5000 adımda platoya girdi. 924 adım/epoch ile bu
epoch 6 civarı — 12 epoch platonun iyice ötesine geçer, 15 epoch'un son epoch'ları boşa gider.

**Dışlananlar** (test setleriyle sızıntı olmasın): Deney 3'ün 5000'i, Deney 4'ün 2500'ü,
eski 80'lik CV testi, yeni 400'lük geniş CV testinin 320 taze örneği.

⚠️ **Değerlendirmeyi `cv_genis_test.ipynb` (400 örnek) ile yap** — 80 örneklik testin gürültüsü
±1.8 puan, bu deneyin kazancını ölçmeye yetmez.

In [ ]:
# 1) Kurulum - torchao'ya HIC dokunmuyoruz (kurmuyoruz, kuruluysa kaldiriyoruz)
!pip install -q evaluate jiwer soundfile transformers peft accelerate datasets
!pip uninstall -y torchao
!apt-get -qq install -y libsndfile1 > /dev/null

In [ ]:
# 2) Drive bagla, proje kodunu ice aktar, ayarlar
import json, os, shutil, sys, random, gc
from copy import deepcopy
from pathlib import Path
from collections import defaultdict, Counter
import torch
import torchaudio as ta
from datasets import load_dataset
from google.colab import drive

INPUT_PATH = "/content/drive/MyDrive/colab_aktarim"
HEDEF_SR = 16000
SEED = 42

# --- Bu deneyin hedefleri ---
N_CV_HEDEF = 8000          # Deney 4'te 2500'du
CV_SECIM_SEED = SEED + 200 # 242 - onceki hicbir secimle cakismayan taze tohum
# FLEURS: hedef yok, train+validation'in TAMAMI (2864) kullanilir

# --- Dislanacak onceki secimleri BIREBIR yeniden uretmek icin gereken sabitler ---
N_DENEY3_CV = 5000         # Deney 3'un egittigi (SEED=42, TUIK)
N_ESKI_TEST = 80           # eski CV testi (TEST_SEED=43)
TEST_SEED = 43
N_DENEY4_CV = 2500         # Deney 4'un egittigi (SEED+100=142, TUIK)
GENIS_TEST_SEED = 4343     # 400'luk genis testin taze 320'si
N_GENIS_TEST = 400

drive.mount('/content/drive')
os.environ["HF_HUB_DISABLE_XET"] = "1"

sys.path.insert(0, f"{INPUT_PATH}/kod")
from egit import trainer_olustur
from egitim_ayarlari import egitim_ayarlari

!df -h /content | tail -1

In [ ]:
# 3) FLEURS (tr_tr train+validation) - TAMAMI, ornekleme yok
print("FLEURS train+validation cekiliyor...")
fleurs_train = load_dataset("parquet", data_files={"train": "hf://datasets/google/fleurs@refs%2Fconvert%2Fparquet/tr_tr/train/0000.parquet"})["train"]
fleurs_val = load_dataset("parquet", data_files={"validation": "hf://datasets/google/fleurs@refs%2Fconvert%2Fparquet/tr_tr/validation/0000.parquet"})["validation"]
print(f"train: {len(fleurs_train)}, validation: {len(fleurs_val)}, TOPLAM: {len(fleurs_train) + len(fleurs_val)}")

FLEURS_SES_DIZIN = Path("/content/veri/fleurs_sesler")
FLEURS_SES_DIZIN.mkdir(parents=True, exist_ok=True)

# NOT: list(fleurs_train) KULLANMIYORUZ - tum sesleri RAM'e almak Deney 3'te tasmaya
# yol acmisti. Dogrudan iterasyon her satiri tek tek cozup birakiyor.
fleurs_satirlar = []
sayac = 0
for ds, ds_adi in ((fleurs_train, "train"), (fleurs_val, "validation")):
    for ornek in ds:
        dalga = torch.tensor(ornek["audio"]["array"], dtype=torch.float32).unsqueeze(0)
        sr = ornek["audio"]["sampling_rate"]
        if sr != HEDEF_SR:
            dalga = ta.functional.resample(dalga, sr, HEDEF_SR)
        metin = ornek.get("raw_transcription") or ornek.get("transcription")
        if not metin or dalga.shape[-1] < HEDEF_SR * 0.5:
            continue

        yol = str(FLEURS_SES_DIZIN / f"fleurs_{sayac}.wav")
        ta.save(yol, dalga, HEDEF_SR, encoding="PCM_S", bits_per_sample=16)

        # NOT: FLEURS'un 'gender' alani ClassLabel (sayisal kod) - str() ile sarmalamazsak
        # CV/ilac verisindeki metin persona ile karisip load_dataset("json", ...)
        # JSON tip tutarsizligindan cokuyor (ArrowInvalid, dogrulandi)
        fleurs_satirlar.append({
            "audio_path": yol,
            "text": metin,
            "sample_rate": HEDEF_SR,
            "duration_s": round(dalga.shape[-1] / HEDEF_SR, 2),
            "dal": "genel_turkce",
            "socrates_asama": "genel",
            "hedef_terim": "genel_turkce",
            "persona": str(ornek.get("gender", "bilinmiyor")),
            "kaynak": "fleurs",
            "ses_profili": f"fleurs_{str(ornek.get('gender', 'bilinmiyor'))}",
        })
        sayac += 1
        if sayac % 500 == 0:
            print(f"  {sayac} FLEURS satiri yazildi")
    gc.collect()

print(f"Toplam gecerli FLEURS satiri: {len(fleurs_satirlar)}")
del fleurs_train, fleurs_val
gc.collect()

In [ ]:
# 4) Common Voice meta-verisi + onceki TUM secimleri disla, taze 8000 sec
CV_REPO = "ysdede/commonvoice_17_tr_fixed"
print("Common Voice (tr) meta-verisi cekiliyor...")
cv_train = load_dataset("parquet", data_files={"train": f"hf://datasets/{CV_REPO}/data/train-00000-of-00001.parquet"})["train"]
cv_validated = load_dataset("parquet", data_files={"validated": f"hf://datasets/{CV_REPO}/data/validated-00000-of-00001.parquet"})["validated"]
print(f"train: {len(cv_train)}, validated: {len(cv_validated)}")

TUIK_YAS_DAGILIMI = {
    "teens": 0.093, "twenties": 0.187, "thirties": 0.183, "fourties": 0.184,
    "fifties": 0.145, "sixties": 0.115, "seventies": 0.067, "eighties": 0.026,
}
CINSIYETLER = ["female_feminine", "male_masculine"]

# Sadece meta-veri kolonlari - ses YOK (RAM guvenligi)
meta_train = [{"kaynak": "train", "idx": i, "age": r["age"], "gender": r["gender"]}
              for i, r in enumerate(cv_train.select_columns(["age", "gender"]))]
meta_validated = [{"kaynak": "validated", "idx": i, "age": r["age"], "gender": r["gender"]}
                  for i, r in enumerate(cv_validated.select_columns(["age", "gender"]))]
etiketli_meta = [r for r in meta_train + meta_validated
                 if r["age"] in TUIK_YAS_DAGILIMI and r["gender"] in CINSIYETLER]
print(f"TUIK kategorileriyle eslesen: {len(etiketli_meta)}/{len(meta_train) + len(meta_validated)}")


def bucketla(havuz):
    b = defaultdict(list)
    for r in havuz:
        b[(r["age"], r["gender"])].append(r)
    return b


def tuik_sec(rnd, havuz, hedef_toplam, uyar=False):
    """Egitim notebook'larindaki inline mantikla BIREBIR ayni - dislama
    hesaplarinin bit-bit ayni sonuc vermesi buna bagli."""
    bucketlar = bucketla(havuz)
    secilen = []
    for yas, yas_agirlik in TUIK_YAS_DAGILIMI.items():
        for cinsiyet in CINSIYETLER:
            hedef_sayi = round(hedef_toplam * yas_agirlik * 0.5)
            mevcut = bucketlar.get((yas, cinsiyet), [])
            secilen.extend(rnd.sample(mevcut, min(hedef_sayi, len(mevcut))))
            if uyar and len(mevcut) < hedef_sayi:
                print(f"  UYARI: {yas}/{cinsiyet} yeterli degil ({len(mevcut)}/{hedef_sayi})")
    eksik = hedef_toplam - len(secilen)
    if eksik > 0:
        secili_id = {id(s) for s in secilen}
        kalanlar = [r for r in havuz if id(r) not in secili_id]
        secilen.extend(rnd.sample(kalanlar, min(eksik, len(kalanlar))))
        if uyar:
            print(f"  TUIK kotalari {eksik} eksik kaldi, rastgele tamamlandi")
    return secilen


def anahtarla(kayitlar):
    return {(r["kaynak"], r["idx"]) for r in kayitlar}


def haric(havuz, *anahtar_kumeleri):
    yasak = set().union(*anahtar_kumeleri) if anahtar_kumeleri else set()
    return [r for r in havuz if (r["kaynak"], r["idx"]) not in yasak]


# 4a) Deney 3'un egittigi 5000
d3 = anahtarla(tuik_sec(random.Random(SEED), etiketli_meta, N_DENEY3_CV))

# 4b) Eski 80'lik CV testi
eski80 = anahtarla(random.Random(TEST_SEED).sample(haric(etiketli_meta, d3), N_ESKI_TEST))

# 4c) Deney 4'un egittigi 2500
d4_havuz = haric(etiketli_meta, d3, eski80)
d4 = anahtarla(tuik_sec(random.Random(SEED + 100), d4_havuz, N_DENEY4_CV))

# 4d) 400'luk genis CV testinin taze 320'si
genis_havuz = haric(etiketli_meta, d3, d4, eski80)
genis320 = anahtarla(random.Random(GENIS_TEST_SEED).sample(genis_havuz, N_GENIS_TEST - N_ESKI_TEST))

print(f"\nDislananlar: Deney3={len(d3)}, Deney4={len(d4)}, eski_test={len(eski80)}, genis_test_ek={len(genis320)}")

# 4e) Bu deney icin: hepsinin DISINDA kalan havuzdan taze 8000
kullanilabilir = haric(etiketli_meta, d3, d4, eski80, genis320)
print(f"Bu deney icin kullanilabilir taze havuz: {len(kullanilabilir)}")

cv_secilen_meta = tuik_sec(random.Random(CV_SECIM_SEED), kullanilabilir, N_CV_HEDEF, uyar=True)
print(f"CV secilen: {len(cv_secilen_meta)}")

# Sizinti kontrolu - secilenler hicbir dislanan kumeyle kesismemeli
secilen_anahtar = anahtarla(cv_secilen_meta)
for ad, kume in (("Deney3", d3), ("Deney4", d4), ("eski_test", eski80), ("genis_test", genis320)):
    kesisim = secilen_anahtar & kume
    print(f"  {ad} ile kesisim: {len(kesisim)}" + ("  <-- SORUN!" if kesisim else "  OK"))

print("\nSecilen yas dagilimi:", dict(Counter(m["age"] for m in cv_secilen_meta)))

In [ ]:
# 5) CV seslerini PARCA PARCA WAV'a cevir (RAM tasmasini onlemek icin)
# Deney 3'te 5000 CV tek seferde list() ile RAM'e alinmis ve oturum tasmisti.
CV_SES_DIZIN = Path("/content/veri/cv_sesler")
CV_SES_DIZIN.mkdir(parents=True, exist_ok=True)
BLOK = 1000

def cv_wav_yaz(dataset, idx_listesi, etiket, baslangic_no):
    satirlar = []
    no = baslangic_no
    for bas in range(0, len(idx_listesi), BLOK):
        parca = dataset.select(idx_listesi[bas:bas + BLOK])
        for ornek in parca:
            dalga = torch.tensor(ornek["audio"]["array"], dtype=torch.float32).unsqueeze(0)
            sr = ornek["audio"]["sampling_rate"]
            if sr != HEDEF_SR:
                dalga = ta.functional.resample(dalga, sr, HEDEF_SR)
            metin = ornek.get("transcription")
            if not metin or dalga.shape[-1] < HEDEF_SR * 0.5:
                continue
            yol = str(CV_SES_DIZIN / f"cv_{no}.wav")
            ta.save(yol, dalga, HEDEF_SR, encoding="PCM_S", bits_per_sample=16)
            satirlar.append({
                "audio_path": yol,
                "text": metin,
                "sample_rate": HEDEF_SR,
                "duration_s": round(dalga.shape[-1] / HEDEF_SR, 2),
                "dal": "genel_turkce",
                "socrates_asama": "genel",
                "hedef_terim": "genel_turkce",
                "persona": str(ornek.get("gender", "bilinmiyor")),
                "kaynak": "common_voice",
                "ses_profili": f"cv_{str(ornek.get('age', 'bilinmiyor'))}_{str(ornek.get('gender', 'bilinmiyor'))}",
            })
            no += 1
        del parca
        gc.collect()
        print(f"  [{etiket}] {min(bas + BLOK, len(idx_listesi))}/{len(idx_listesi)} -> {len(satirlar)} gecerli satir")
    return satirlar

cv_train_idx = sorted(m["idx"] for m in cv_secilen_meta if m["kaynak"] == "train")
cv_validated_idx = sorted(m["idx"] for m in cv_secilen_meta if m["kaynak"] == "validated")
print(f"train'den {len(cv_train_idx)}, validated'den {len(cv_validated_idx)} kayit yazilacak")

cv_satirlar = cv_wav_yaz(cv_train, cv_train_idx, "cv-train", 0)
cv_satirlar += cv_wav_yaz(cv_validated, cv_validated_idx, "cv-validated", len(cv_satirlar))
print(f"\nToplam gecerli CV satiri: {len(cv_satirlar)}")

del cv_train, cv_validated
gc.collect()

# Drive'a yedek - egitim koparsa veriyi tekrar cekmek gerekmesin diye
os.makedirs(f"{INPUT_PATH}/data", exist_ok=True)
GENEL_JSONL_DRIVE = f"{INPUT_PATH}/data/genel_turkce_8k_train.jsonl"
with open(GENEL_JSONL_DRIVE, "w", encoding="utf-8") as f:
    for r in (fleurs_satirlar + cv_satirlar):
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Yazildi (Drive yedek): {GENEL_JSONL_DRIVE}")
!df -h /content | tail -1

In [ ]:
# 6) Ilac verisini hazirla (boyut butunluk kontroluyle), hepsini birlestir
YEREL_VERI = Path("/content/veri")
YEREL_SESLER = YEREL_VERI / "sesler"
YEREL_SESLER.mkdir(parents=True, exist_ok=True)

def ilac_hazirla(kaynak_jsonl, etiket, zorunlu=True):
    if not os.path.exists(kaynak_jsonl):
        mesaj = f"  [{etiket}] {kaynak_jsonl} Drive'da YOK"
        if zorunlu:
            raise FileNotFoundError(mesaj)
        print(mesaj + " - bu veri ATLANIYOR")
        return []
    satirlar = [json.loads(l) for l in open(kaynak_jsonl, encoding="utf-8")]
    kopyalanan = 0
    eksik_ses = []
    gecerli_satirlar = []
    for i, r in enumerate(satirlar):
        fname = os.path.basename(r["audio_path"].replace("\\", "/"))
        kaynak = Path(INPUT_PATH) / "data" / "sesler" / fname
        hedef = YEREL_SESLER / fname
        if not kaynak.exists():
            eksik_ses.append(fname)
            continue
        # NOT: sadece exists() bakmak yetmez - yarim kalmis bir kopya sessizce
        # kalir ve egitimin ortasinda "Invalid data found" ile cokertir (yasandi)
        gecerli = hedef.exists() and hedef.stat().st_size == kaynak.stat().st_size
        if not gecerli:
            shutil.copy(kaynak, hedef)
            kopyalanan += 1
        r["audio_path"] = str(hedef)
        gecerli_satirlar.append(r)
        if (i + 1) % 1000 == 0:
            print(f"  [{etiket}] {i + 1}/{len(satirlar)} ({kopyalanan} yeni/duzeltilen)")
    if eksik_ses:
        print(f"  [{etiket}] UYARI: {len(eksik_ses)} ses dosyasi Drive'da yok, o satirlar atlandi")
        print(f"    ornek: {eksik_ses[:3]}")
        print(f"    cozum: veri_uretimi/cikti/sesler/ icindeki bu dosyalari {INPUT_PATH}/data/sesler/ altina yukle")
    print(f"  [{etiket}] tamamlandi: {len(gecerli_satirlar)}/{len(satirlar)} satir, {kopyalanan} yeni/duzeltilen kopya")
    return gecerli_satirlar

print("Ilac train verisi hazirlaniyor...")
ilac_satirlar = ilac_hazirla(f"{INPUT_PATH}/data/ilac_train.jsonl", "ilac-train")
print("Acil tip train verisi hazirlaniyor (val+test EGITIME GIRMEZ, 4. eksen olarak ayri)...")
acil_satirlar = ilac_hazirla(f"{INPUT_PATH}/data/acil_tip_train.jsonl", "acil_tip-train", zorunlu=False)
print("Val verisi hazirlaniyor (SADECE ilac - Deney 1-4 ile karsilastirilabilir kalsin)...")
val_satirlar_ham = ilac_hazirla(f"{INPUT_PATH}/data/ilac_val.jsonl", "val")

with open(YEREL_VERI / "ilac_val.jsonl", "w", encoding="utf-8") as f:
    for r in val_satirlar_ham:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

birlesik = ilac_satirlar + acil_satirlar + fleurs_satirlar + cv_satirlar
random.Random(SEED).shuffle(birlesik)
TRAIN_DOSYASI = str(YEREL_VERI / "ilac_genel8k_train.jsonl")
with open(TRAIN_DOSYASI, "w", encoding="utf-8") as f:
    for r in birlesik:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

n_train = len(birlesik)
n_genel = len(fleurs_satirlar) + len(cv_satirlar)
print(f"\nBIRLESIK VERI: ilac={len(ilac_satirlar)}, acil_tip={len(acil_satirlar)}, fleurs={len(fleurs_satirlar)}, cv={len(cv_satirlar)}, toplam={n_train}")
print(f"genel turkce orani: %{100 * n_genel / n_train:.1f}")

# persona alani tip tutarliligi kontrolu - hepsi string olmali (ArrowInvalid'i onler)
tipler = Counter(type(r["persona"]).__name__ for r in birlesik)
print(f"persona alan tipleri: {dict(tipler)}" + ("  OK" if set(tipler) == {"str"} else "  <-- SORUN!"))

# toplam ses suresi
toplam_saat = sum(r.get("duration_s", 0) for r in birlesik) / 3600
print(f"Toplam ses suresi: {toplam_saat:.2f} saat")

In [ ]:
# 7) Ayarlar - r=32/alpha=64 (Deney 4 ile AYNI), 12 epoch, egitimi baslat
EFEKTIF_BATCH = 16
EPOCH_TAVANI = 12
ADIM_BASINA_EPOCH = -(-n_train // EFEKTIF_BATCH)
TOPLAM_ADIM = ADIM_BASINA_EPOCH * EPOCH_TAVANI
WARMUP_STEPS = round(TOPLAM_ADIM * 0.10)

LORA_R = 32
LORA_ALPHA = 64

ayarlar = deepcopy(egitim_ayarlari)
ayarlar.output_dir = f"{INPUT_PATH}/checkpoints/ilac_genel8k_r32"
ayarlar.num_train_epochs = EPOCH_TAVANI
ayarlar.warmup_steps = WARMUP_STEPS
ayarlar.fp16 = torch.cuda.is_available()
ayarlar.save_total_limit = None
ayarlar.report_to = []

print(f"adim/epoch: {ADIM_BASINA_EPOCH}, toplam adim: {TOPLAM_ADIM}, warmup: {WARMUP_STEPS}, fp16: {ayarlar.fp16}")
print(f"LoRA r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Tahmini sure: {TOPLAM_ADIM / 1759:.1f} saat (Deney 4'un gozlenen ~1759 adim/saat hizina gore)")

VAL_DOSYASI = str(YEREL_VERI / "ilac_val.jsonl")

trainer, peft_model, processor = trainer_olustur(
    ayarlar=ayarlar,
    lora_dropout=0.05,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    erken_durdurma_sabri=None,
    train_dosyasi=TRAIN_DOSYASI,
    val_dosyasi=VAL_DOSYASI,
    gurultu_p=0.45, reverb_p=0.25, birlestir_p=0.0, kazanc_p=0.5,
    veri_aciklamasi=f"ilac_genel8k_train.jsonl ({n_train} satir: ilac={len(ilac_satirlar)}, acil_tip={len(acil_satirlar)}, fleurs={len(fleurs_satirlar)}, cv={len(cv_satirlar)}) - r={LORA_R}/alpha={LORA_ALPHA}, {EPOCH_TAVANI} epoch, val=SADECE ilac",
)
peft_model.print_trainable_parameters()
print(f"train: {len(trainer.train_dataset)}, val: {len(trainer.eval_dataset)}")

devam_et = None
if os.path.isdir(ayarlar.output_dir):
    checkpointler = [d for d in os.listdir(ayarlar.output_dir) if d.startswith("checkpoint-")]
    if checkpointler:
        devam_et = True
        print(f"Onceki checkpoint bulundu ({len(checkpointler)} adet) - oradan devam edilecek.")

print(f"\n=== EGITIM BASLIYOR (tahmini ~{TOPLAM_ADIM / 1759:.1f} saat) ===")
trainer.train(resume_from_checkpoint=devam_et)